In [192]:
import pandas as pd 
import matplotlib.pyplot as plt
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

In [193]:
class WindowDataset(Dataset):
    def __init__(self, _data, _window):
        # _data : tensor, array 1차원 데이터 형태
        # _window : 구간의 크기 
        self.data = _data
        self.window = _window
        # DataLoader에서 사용이 가능한 인덱스의 최대 값
        self.n = len(_data) - _window

    def __len__(self):
        return self.n
    
    def __getitem__(self, idx):
        # idx : 0 ~ self.n-1 사이의 정수가 대입 (DataLoader에서 자동으로 대입)
        x = self.data[idx:idx + self.window]
        y = self.data[idx + self.window]
        return x, y
    

In [ ]:
# RNN 모델 정의 
class RNNModel(nn.Module):
    def __init__(self,
                 input_size, 
                 hidden_size = 64, 
                 num_layers = 1, 
                 dropout = 0.0, 
                 nonlinearity = 'tanh', 
                 bidirectional = False):
        super(RNNModel, self).__init__()
        # super().__init__()
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size, 
            num_layers = num_layers, 
            dropout = dropout, 
            nonlinearity = nonlinearity, 
            bidirectional = bidirectional, 
            # batch_first는 기본이 Fasle였던걸 까먹고 안 넣고 있었네요 
            # 죄송합니다. 
            batch_first = True
        )

        # output_feature가 역방향을 포함한다면 2배로 늘어난다. 
        if bidirectional:
            hidden_size *= 2
        print(f"hidden_size : {hidden_size}")
        self.model = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, h_n = self.rnn(x)
        last_hidden = h_n[-1]
        result = self.model(last_hidden)
        return result

In [ ]:
# 모델 학습 시 검증 데이터를 이용하여 모델의 성능을 평가 할수 있도록 검증 데이터 평가 함수 

@torch.no_grad()
def evaluate_mse(dataloader, model):
    # dataloader -> 검증 데이터셋의 dataloader
    model.eval()
    total_loss = 0
    total_n = 0
    for x, y in dataloader:
        x = x.float()
        y = y.float()
        pred = model(x)
        loss = nn.MSELoss()(pred, y)
        # total_loss에서도 +=을 +로만 쓰는 막대한 실수를 했네요 
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)
    return total_loss / max(total_n, 1)

In [196]:
df = pd.read_csv("../csv/aapl.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9715 entries, 0 to 9714
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       9715 non-null   object 
 1   Open       9714 non-null   float64
 2   High       9714 non-null   float64
 3   Low        9714 non-null   float64
 4   Close      9714 non-null   float64
 5   Adj Close  9714 non-null   float64
 6   Volume     9713 non-null   float64
dtypes: float64(6), object(1)
memory usage: 531.4+ KB


In [197]:
df.dropna(inplace=True)

In [198]:
# 수정종가, 날짜 데이터만 추출
df = df[['Date', 'Adj Close']]

In [199]:
values = df[['Adj Close']].values
values

array([[  0.410525],
       [  0.389106],
       [  0.360548],
       ...,
       [199.460007],
       [198.779999],
       [199.169998]], shape=(9713, 1))

In [200]:
df['Adj Close'].values.reshape(-1, 1)

array([[  0.410525],
       [  0.389106],
       [  0.360548],
       ...,
       [199.460007],
       [198.779999],
       [199.169998]], shape=(9713, 1))

In [201]:
# 75:25 비율로 학습, 검증 데이터로 분할
split_idx = int(len(values) * 0.75)
split_idx

7284

In [202]:
train_data = values[:split_idx]
test_data = values[split_idx:]

In [203]:
# MinMaxScaler로 데이터 정규화
scaler = MinMaxScaler()
train_sc = scaler.fit_transform(train_data)
test_sc = scaler.transform(test_data)

In [204]:
test_sc

array([[0.93723805],
       [0.956606  ],
       [0.91811525],
       ...,
       [7.80873011],
       [7.78208726],
       [7.7973675 ]], shape=(2429, 1))

In [205]:
# 스케일링이 완료된 데이터를 Tensor로 변환 
train_sc = torch.tensor(train_sc, dtype=torch.float32)
test_sc = torch.tensor(test_sc, dtype=torch.float32)

In [206]:
# WindowDataset에 데이터를 대입 
train_ds = WindowDataset(train_sc, _window=60)
test_ds = WindowDataset(test_sc, _window=60)

In [207]:
# Dataset를 DataLoader로 생성 
train_dl = DataLoader(train_ds, batch_size=128, shuffle = True, drop_last = True)
test_dl = DataLoader(test_ds, batch_size=256, shuffle=False, drop_last = False)

In [208]:
aapl_model = RNNModel(
    input_size = 1
)
criterion = nn.MSELoss()
optimizer = optim.Adam(aapl_model.parameters(), lr = 0.001)

hidden_size : 64


In [209]:
# 모델 학습 
train_history, test_history = [], []
for epoch in range(20):
    aapl_model.train()
    running, n_seen = 0.0, 0
    for x, y in train_dl:
        x = x.float()
        y = y.float()
        pred = aapl_model(x)
        loss = criterion(pred, y)
        # print(x.shape, y.shape)
        # print(pred.shape, y.shape)
        # break
        optimizer.zero_grad()
        loss.backward()
        # 가중치 발산 방지 
        nn.utils.clip_grad_norm_(aapl_model.parameters(), 1.0)
        optimizer.step()

        running += loss.item() * y.size(0)
        n_seen += y.size(0)
    train_mse = running / n_seen
    test_mse = evaluate_mse(test_dl, aapl_model)
    train_history.append(train_mse)
    test_history.append(test_mse)
    print(f"Epoch {epoch+1} Train MSE: {round(train_mse, 8)} Test MSE: {round(test_mse, 8)}")

Epoch 1 Train MSE: 0.01067803 Test MSE: 7.99229292
Epoch 2 Train MSE: 0.00019688 Test MSE: 7.24698329
Epoch 3 Train MSE: 0.00013779 Test MSE: 6.87214939
Epoch 4 Train MSE: 0.00012366 Test MSE: 6.57960244
Epoch 5 Train MSE: 0.00013356 Test MSE: 6.34080545
Epoch 6 Train MSE: 9.91e-05 Test MSE: 6.0768194
Epoch 7 Train MSE: 0.00011086 Test MSE: 5.94852005
Epoch 8 Train MSE: 0.00010893 Test MSE: 5.76338811
Epoch 9 Train MSE: 9.246e-05 Test MSE: 5.62057401
Epoch 10 Train MSE: 9.361e-05 Test MSE: 5.53371648
Epoch 11 Train MSE: 0.00010419 Test MSE: 5.42805964
Epoch 12 Train MSE: 0.00010709 Test MSE: 5.29544225
Epoch 13 Train MSE: 9.457e-05 Test MSE: 5.21784018
Epoch 14 Train MSE: 8.853e-05 Test MSE: 5.09659018
Epoch 15 Train MSE: 9.264e-05 Test MSE: 5.01730779
Epoch 16 Train MSE: 8.751e-05 Test MSE: 4.9184638
Epoch 17 Train MSE: 9.219e-05 Test MSE: 4.87384405
Epoch 18 Train MSE: 8.198e-05 Test MSE: 4.72725614
Epoch 19 Train MSE: 7.787e-05 Test MSE: 4.67354362
Epoch 20 Train MSE: 7.544e-05 Test